In [1]:
import pandas as pd
import numpy as np
import locale
import traceback
import os
import sys
import tkinter as tk
from tkinter import filedialog

# IMPORTANTE: Verificación de xlwings
try:
    import xlwings as xw
except ImportError:
    print("ERROR CRÍTICO: Necesitas instalar la librería 'xlwings'.")
    input("Presiona ENTER para salir...")
    sys.exit()

# --- Nombres de Hojas ---
NOMBRE_HOJA_MP_CONC = "MP Conciliado"
NOMBRE_HOJA_SISTEMA_CONC = "Sistema Conciliado"
NOMBRE_HOJA_VENTAS_SISTEMA = "Ventas_Sistema"
NOMBRE_HOJA_CAJA_ADICION = "Caja Adicion"
NOMBRE_HOJA_DESTINO = "Arqueo_por_Turno"

# --- Columnas ---
COL_MONTO_MP = 'VALOR DE LA COMPRA'
COL_MONTO_SISTEMA = 'Monto Bruto pago'
COL_FECHA_MP = 'FECHA DE ORIGEN'
COL_FECHA_SISTEMA = 'Fecha'
COL_ESTADO = 'Estado'
COL_PLATAFORMA_SISTEMA = "Medio de cobro"
COL_TURNO = "TURNO"
COL_PROPINA = "Propina"
COL_DESCUENTOS = "Descuentos"
COL_BOLETA = "Boleta"
COL_COMENSALES = "Cantidad de comensales"
COL_RECUENTO_EFECTIVO = "Recuento Efectivo"
COL_APERTURA_CAJA = "APERTURA CAJA Efectivo"
COL_MONTO_CAJA = "Monto"
COL_FECHA_CAJA = "Fecha Modificación"
COL_TIPO_CAJA = "Proveedor / Para"  # <--- NUEVO: Para distinguir Ingreso/Egreso

VALOR_PLATAFORMA_MP = "Mercado Pago"
VALOR_PLATAFORMA_EFECTIVO = "Efectivo"
VALOR_PLATAFORMA_CTACTE = "Cta Cte"
VALOR_INGRESO_CAJA = "Ingreso de Dinero" # <--- Texto exacto en Excel
VALOR_EGRESO_CAJA = "Egreso de Dinero"   # <--- Texto exacto en Excel

# --- Listas de Columnas para Formato ---
COLS_FORMATO_PORCENTAJE = [
    '% diferencia facturacion', 
    '% diferencia MP',
    '% diferencia efectivo'
]

COLS_FORMATO_NUMERO = [
    'Ventas Totales', 'Cantidad de comensales', 'Comensales promedio', 'Cantidad de ticket', 
    'Ticket Promedio', 'Propina', 'Ventas MP Real', 'Ventas Sistema MP Real', 'Ventas Sistema MP Correcto',
    'Conciliado MP REAL', 'No conciliado MP REAL', 'Conciliado Sistema MP', 
    'No conciiado Sistema MP', 'Diferencia MP sistema vs real', 'Efectivo Total', 
    'Conciliado Efectivo, Sistema', 'Efectivo Real', 'Cta Cte Total', 
    'Conciliado CtaCte, Sistema', 'Cta Cte Real', 'Otros',  # <--- CORREGIDO: Se agregó la coma que faltaba
    'Ideal facturacion', 'Diferencia de facturacion',
    'Recuento Efectivo', 'Pagos en Efectivo', 'Recuento sistema efectivo', 'APERTURA CAJA Efectivo',
    'Diferencia Efectivo',
    'Ingreso de Dinero'
]

def seleccionar_archivo(titulo_ventana):
    print(f"--> Esperando selección de: {titulo_ventana}...")
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    ruta = filedialog.askopenfilename(title=titulo_ventana, filetypes=[("Archivos de Excel", "*.xlsx;*.xlsm")])
    root.destroy()
    if not ruta:
        print("   X Selección cancelada por el usuario.")
        return None
    print(f"   V Seleccionado: {os.path.basename(ruta)}")
    return ruta

def get_column_letter(idx):
    letters = []
    while idx > 0:
        idx, remainder = divmod(idx - 1, 26)
        letters.append(chr(65 + remainder))
    return ''.join(reversed(letters))

def limpiar_turno(valor):
    if pd.isna(valor): return "SIN_TURNO"
    s = str(valor).strip()
    if s.endswith('.0'): s = s[:-2]
    return s

def limpiar_monto_argentina(valor):
    if pd.isna(valor): return 0.0
    if isinstance(valor, (int, float)): return float(valor)
    s = str(valor).strip().replace('$', '').replace(' ', '')
    if ',' in s and '.' in s: s = s.replace('.', '').replace(',', '.')
    elif ',' in s: s = s.replace(',', '.')
    try: return float(s)
    except ValueError: return 0.0

def clasificar_estado_diferencia(valor):
    val_rounded = round(valor, 2)
    if val_rounded < 0: return "Faltante"
    elif val_rounded > 0: return "Sobrante"
    return ""

def generar_arqueo_por_turno():
    print("=================================================")
    print("   GENERADOR DE ARQUEO POR TURNO (V12 - SIN GETNET)")
    print("=================================================\n")

    ruta_turnos = seleccionar_archivo("1. Selecciona el archivo TURNOS PROCESADOS")
    if not ruta_turnos: return

    ruta_conciliacion = seleccionar_archivo("2. Selecciona el archivo RESULTADO CONCILIACION")
    if not ruta_conciliacion: return

    ruta_informes = seleccionar_archivo("3. Selecciona el archivo INFORMES PARADOR (Ventas Sistema)")
    if not ruta_informes: return

    ruta_caja = seleccionar_archivo("4. Selecciona el archivo con la hoja CAJA ADICION (Macro)")
    if not ruta_caja: return

    print("\nPASO 2: Selecciona el archivo de DESTINO.")
    ruta_salida = seleccionar_archivo("5. Selecciona el archivo ARQUEO FINAL (Excel .xlsm)")
    if not ruta_salida: return

    try:
        try: locale.setlocale(locale.LC_TIME, 'es_ES.UTF-8')
        except: pass

        print("\n3. Leyendo archivos de datos...")
        try:
            df_turnos_base = pd.read_excel(ruta_turnos, sheet_name=0)
            df_mp_conc = pd.read_excel(ruta_conciliacion, sheet_name=NOMBRE_HOJA_MP_CONC)
            df_sistema_conc = pd.read_excel(ruta_conciliacion, sheet_name=NOMBRE_HOJA_SISTEMA_CONC)
            df_ventas_extra = pd.read_excel(ruta_informes, sheet_name=NOMBRE_HOJA_VENTAS_SISTEMA)
            df_caja_adicion = pd.read_excel(ruta_caja, sheet_name=NOMBRE_HOJA_CAJA_ADICION)
        except Exception as e:
            print(f"\nERROR AL LEER EXCEL: {e}")
            return

        print("   -> Datos cargados. Procesando...")

        if 'Apertura' not in df_turnos_base.columns:
            f_ap = pd.to_datetime(df_turnos_base['Fecha Apertura'], dayfirst=True, errors='coerce')
            f_ci = pd.to_datetime(df_turnos_base['Fecha Cierre'], dayfirst=True, errors='coerce')
            h_ap = df_turnos_base['Hs Ap. Caja'].astype(str).str.strip()
            h_ci = df_turnos_base['Hs Cierre Caja'].astype(str).str.strip()
            df_turnos_base['Apertura'] = pd.to_datetime(f_ap.dt.strftime('%Y-%m-%d') + ' ' + h_ap, errors='coerce')
            df_turnos_base['Cierre'] = pd.to_datetime(f_ci.dt.strftime('%Y-%m-%d') + ' ' + h_ci, errors='coerce')
        else:
            df_turnos_base['Apertura'] = pd.to_datetime(df_turnos_base['Apertura'], dayfirst=True, errors='coerce')
            df_turnos_base['Cierre'] = pd.to_datetime(df_turnos_base['Cierre'], dayfirst=True, errors='coerce')

        df_turnos_base['Fecha'] = pd.to_datetime(df_turnos_base['Fecha Apertura'], dayfirst=True, errors='coerce').dt.date
        
        if COL_RECUENTO_EFECTIVO not in df_turnos_base.columns:
            df_turnos_base[COL_RECUENTO_EFECTIVO] = 0.0
        else:
            df_turnos_base[COL_RECUENTO_EFECTIVO] = pd.to_numeric(df_turnos_base[COL_RECUENTO_EFECTIVO], errors='coerce').fillna(0)

        if COL_APERTURA_CAJA not in df_turnos_base.columns:
            df_turnos_base[COL_APERTURA_CAJA] = 0.0
        else:
            df_turnos_base[COL_APERTURA_CAJA] = pd.to_numeric(df_turnos_base[COL_APERTURA_CAJA], errors='coerce').fillna(0)

        if COL_COMENSALES not in df_turnos_base.columns: df_turnos_base[COL_COMENSALES] = 0
        else: df_turnos_base[COL_COMENSALES] = pd.to_numeric(df_turnos_base[COL_COMENSALES], errors='coerce').fillna(0)

        dias_espanol = {0:'Lunes', 1:'Martes', 2:'Miércoles', 3:'Jueves', 4:'Viernes', 5:'Sábado', 6:'Domingo'}
        df_turnos_base['Dia'] = pd.to_datetime(df_turnos_base['Fecha']).dt.dayofweek.map(dias_espanol)
        df_turnos_base['Hora Apertura'] = df_turnos_base['Apertura']
        df_turnos_base['Hora Cierre'] = df_turnos_base['Cierre']

        # --- CORREGIDO: Se eliminó df_getnet_conc de la lista ---
        for df in [df_turnos_base, df_mp_conc, df_sistema_conc, df_ventas_extra, df_caja_adicion]:
            if COL_TURNO in df.columns: df[COL_TURNO] = df[COL_TURNO].apply(limpiar_turno)

        def prep(df, c_m, c_f):
            df['Monto_Num'] = pd.to_numeric(df[c_m], errors='coerce').fillna(0)
            df['datetime_col'] = pd.to_datetime(df[c_f], dayfirst=True, errors='coerce')
            return df
        
        df_mp_conc = prep(df_mp_conc, COL_MONTO_MP, COL_FECHA_MP)
        df_sistema_conc = prep(df_sistema_conc, COL_MONTO_SISTEMA, COL_FECHA_SISTEMA)
        df_caja_adicion = prep(df_caja_adicion, COL_MONTO_CAJA, COL_FECHA_CAJA) 

        col_filtro = 'Clasificacion'
        if col_filtro in df_mp_conc.columns:
            estado_norm = df_mp_conc[COL_ESTADO].fillna('').astype(str).str.strip()
            clasif_norm = df_mp_conc[col_filtro].fillna('').astype(str).str.strip().str.lower()
            mask_excluir = (estado_norm == 'No Conciliado') & (clasif_norm == 'interno')
            if mask_excluir.sum() > 0: df_mp_conc = df_mp_conc[~mask_excluir]

        if COL_PLATAFORMA_SISTEMA in df_sistema_conc.columns:
            df_sistema_conc[COL_PLATAFORMA_SISTEMA] = df_sistema_conc[COL_PLATAFORMA_SISTEMA].astype(str).str.strip()

        df_ventas_extra['datetime_col'] = pd.to_datetime(df_ventas_extra[COL_FECHA_SISTEMA], dayfirst=True, errors='coerce')
        for c in [COL_PROPINA, COL_DESCUENTOS, COL_BOLETA]:
            if c not in df_ventas_extra.columns: df_ventas_extra[c] = 0
            df_ventas_extra[c] = df_ventas_extra[c].apply(limpiar_monto_argentina)

        cols_extra = ['Encargado']
        for c in cols_extra:
            if c not in df_turnos_base.columns: df_turnos_base[c] = "" 

        df_arqueo = df_turnos_base.copy()
        df_arqueo['Horas Trabajadas'] = df_arqueo['Cierre'] - df_arqueo['Apertura']

        cols_calc = [
            'Ventas MP Real', 'Ventas Sistema MP Real', 'Conciliado MP REAL', 'No conciliado MP REAL',
            'Conciliado Sistema MP', 'No conciiado Sistema MP', 'Diferencia MP sistema vs real',
            'Efectivo Total', 'Conciliado Efectivo, Sistema', 'Efectivo Real', 
            'Cta Cte Total', 'Conciliado CtaCte, Sistema', 'Cta Cte Real', 
            'Otros',
            'Propina', 'Descuentos', 'Ventas facturadas', 'Ideal facturacion', 'Diferencia de facturacion',
            'Ventas Totales', 'Cantidad de ticket', 'Comensales promedio', 'Ticket Promedio',
            'Pagos en Efectivo', 'Recuento sistema efectivo',
            'Diferencia Efectivo', '% diferencia efectivo',
            'Ingreso de Dinero'
        ]
        for col in cols_calc: df_arqueo[col] = 0.0
        
        df_arqueo['Hs Primera Venta Sistema'] = pd.NaT
        df_arqueo['Hs Ultima Venta Sistema'] = pd.NaT
        col_horas = [f"{h:02d}:00 - {h+1:02d}:00" for h in range(24)]
        for h in col_horas: df_arqueo[h] = 0.0

        print("4. Realizando cruces y cálculos...")
        medios = [VALOR_PLATAFORMA_MP, VALOR_PLATAFORMA_EFECTIVO, VALOR_PLATAFORMA_CTACTE]

        for i, row in df_arqueo.iterrows():
            st, et, turno = row['Apertura'], row['Cierre'], row[COL_TURNO]
            if pd.isna(st) or pd.isna(et): continue

            msk_sis = (df_sistema_conc['datetime_col'] >= st) & (df_sistema_conc['datetime_col'] <= et) & (df_sistema_conc[COL_TURNO] == turno)
            msk_mp = (df_mp_conc['datetime_col'] >= st) & (df_mp_conc['datetime_col'] <= et) & (df_mp_conc[COL_TURNO] == turno)
            msk_ex = (df_ventas_extra['datetime_col'] >= st) & (df_ventas_extra['datetime_col'] <= et) & (df_ventas_extra[COL_TURNO] == turno)
            msk_caja = (df_caja_adicion['datetime_col'] >= st) & (df_caja_adicion['datetime_col'] <= et) & (df_caja_adicion[COL_TURNO] == turno)

            d_sis = df_sistema_conc[msk_sis]
            d_mp = df_mp_conc[msk_mp]
            d_ex = df_ventas_extra[msk_ex]
            d_caja = df_caja_adicion[msk_caja]

            df_arqueo.at[i, 'Cantidad de ticket'] = d_sis[d_sis[COL_PLATAFORMA_SISTEMA].isin(medios)].shape[0]

            # --- LÓGICA DE CAJA (INGRESO VS EGRESO) ---
            val_ingreso = 0.0
            val_egreso = 0.0

            if COL_TIPO_CAJA in d_caja.columns:
                # Normalizamos texto para evitar errores de espacios o mayúsculas
                tipo_normalizado = d_caja[COL_TIPO_CAJA].astype(str).str.strip()
                
                # Sumar ingresos
                val_ingreso = d_caja[tipo_normalizado == VALOR_INGRESO_CAJA]['Monto_Num'].sum()
                # Sumar egresos
                val_egreso = d_caja[tipo_normalizado == VALOR_EGRESO_CAJA]['Monto_Num'].sum()
            else:
                # Si no existe la columna, asumimos comportamiento anterior (todo es pago)
                val_egreso = d_caja['Monto_Num'].sum()

            df_arqueo.at[i, 'Ingreso de Dinero'] = val_ingreso
            df_arqueo.at[i, 'Pagos en Efectivo'] = val_egreso
            # -------------------------------------------

            # =================================================================
            # CALCULO VENTAS POR HORA
            # =================================================================
            mask_ef_no_conc = (d_sis[COL_PLATAFORMA_SISTEMA] == VALOR_PLATAFORMA_EFECTIVO) & (d_sis[COL_ESTADO] != 'Conciliado')
            d_ef_no_conc = d_sis[mask_ef_no_conc]
            mask_cc_no_conc = (d_sis[COL_PLATAFORMA_SISTEMA] == VALOR_PLATAFORMA_CTACTE) & (d_sis[COL_ESTADO] != 'Conciliado')
            d_cc_no_conc = d_sis[mask_cc_no_conc]

            # --- CORREGIDO: Se eliminó d_gn de la concatenación ---
            d_h = pd.concat([d_mp[['datetime_col','Monto_Num']], d_ef_no_conc[['datetime_col','Monto_Num']], d_cc_no_conc[['datetime_col','Monto_Num']]], ignore_index=True)
            
            ventas_brutas_hora = {}
            if not d_h.empty:
                d_h['h'] = d_h['datetime_col'].dt.hour
                ventas_brutas_hora = d_h.groupby('h')['Monto_Num'].sum().to_dict()

            propinas_hora = {}
            if not d_ex.empty and COL_PROPINA in d_ex.columns:
                d_ex_temp = d_ex.copy()
                d_ex_temp['h'] = d_ex_temp['datetime_col'].dt.hour
                propinas_hora = d_ex_temp.groupby('h')[COL_PROPINA].sum().to_dict()

            for h_idx in range(24):
                bruto = ventas_brutas_hora.get(h_idx, 0.0)
                propina_h = propinas_hora.get(h_idx, 0.0)
                neto_h = bruto - propina_h
                if bruto != 0 or propina_h != 0:
                    df_arqueo.at[i, col_horas[h_idx]] = neto_h

            propina_val = d_ex[COL_PROPINA].sum()
            df_arqueo.at[i, 'Propina'] = propina_val
            df_arqueo.at[i, 'Descuentos'] = d_ex[COL_DESCUENTOS].sum()
            fact_tot = d_ex[COL_BOLETA].sum()
            df_arqueo.at[i, 'Ventas facturadas'] = fact_tot

            if not d_sis.empty:
                df_arqueo.at[i, 'Hs Primera Venta Sistema'] = d_sis['datetime_col'].min()
                df_arqueo.at[i, 'Hs Ultima Venta Sistema'] = d_sis['datetime_col'].max()

            # --- TOTALES PLATAFORMAS (REALES) ---
            mp_real = d_mp['Monto_Num'].sum()
            
            df_arqueo.at[i, 'Ventas MP Real'] = mp_real
            df_arqueo.at[i, 'Conciliado MP REAL'] = d_mp[d_mp[COL_ESTADO]=='Conciliado']['Monto_Num'].sum()
            df_arqueo.at[i, 'No conciliado MP REAL'] = d_mp[d_mp[COL_ESTADO]!='Conciliado']['Monto_Num'].sum()

            ideal = mp_real 
            df_arqueo.at[i, 'Ideal facturacion'] = ideal
            df_arqueo.at[i, 'Diferencia de facturacion'] = fact_tot - ideal

            # --- TOTALES SISTEMA (RAW) ---
            s_mp = d_sis[d_sis[COL_PLATAFORMA_SISTEMA] == VALOR_PLATAFORMA_MP]
            # --- CORREGIDO: Se eliminó s_gn ---

            raw_sis_mp = s_mp['Monto_Num'].sum()

            df_arqueo.at[i, 'Ventas Sistema MP Real'] = raw_sis_mp
            df_arqueo.at[i, 'Conciliado Sistema MP'] = s_mp[s_mp[COL_ESTADO]=='Conciliado']['Monto_Num'].sum()
            df_arqueo.at[i, 'No conciiado Sistema MP'] = s_mp[s_mp[COL_ESTADO]!='Conciliado']['Monto_Num'].sum()

            ef_tot = d_sis[d_sis[COL_PLATAFORMA_SISTEMA] == VALOR_PLATAFORMA_EFECTIVO]['Monto_Num'].sum()
            ef_conc = d_sis[(d_sis[COL_PLATAFORMA_SISTEMA] == VALOR_PLATAFORMA_EFECTIVO) & (d_sis[COL_ESTADO]=='Conciliado')]['Monto_Num'].sum()
            df_arqueo.at[i, 'Efectivo Total'] = ef_tot
            df_arqueo.at[i, 'Conciliado Efectivo, Sistema'] = ef_conc
            
            # Efectivo Real = (Total Sist - Conciliado Sist) + Ingreso de Dinero
            dif_ef = ef_tot - ef_conc 
            df_arqueo.at[i, 'Efectivo Real'] = dif_ef + val_ingreso

            cc_tot = d_sis[d_sis[COL_PLATAFORMA_SISTEMA] == VALOR_PLATAFORMA_CTACTE]['Monto_Num'].sum()
            cc_conc = d_sis[(d_sis[COL_PLATAFORMA_SISTEMA] == VALOR_PLATAFORMA_CTACTE) & (d_sis[COL_ESTADO]=='Conciliado')]['Monto_Num'].sum()
            df_arqueo.at[i, 'Cta Cte Total'] = cc_tot
            df_arqueo.at[i, 'Conciliado CtaCte, Sistema'] = cc_conc
            dif_cc = cc_tot - cc_conc
            df_arqueo.at[i, 'Cta Cte Real'] = dif_cc

            otros = d_sis[(~d_sis[COL_PLATAFORMA_SISTEMA].isin(medios)) & (d_sis[COL_ESTADO]!='Conciliado')]['Monto_Num'].sum()
            df_arqueo.at[i, 'Otros'] = otros

            # --- CORREGIDO: Se eliminó gn_real del cálculo de Ventas Totales ---
            df_arqueo.at[i, 'Ventas Totales'] = mp_real + dif_ef + dif_cc - propina_val
            
            df_arqueo.at[i, 'Diferencia MP sistema vs real'] = df_arqueo.at[i, 'No conciliado MP REAL'] - df_arqueo.at[i, 'No conciiado Sistema MP']
            # --- CORREGIDO: Se eliminó el cálculo de Diferencia Getnet ---

        # =========================================================================
        # CALCULO DE EFECTIVO
        # =========================================================================
        # Recuento Sistema = Apertura + (Ventas Efectivo + Ingresos Extra) - Pagos/Egresos
        df_arqueo['Recuento sistema efectivo'] = (
            df_arqueo[COL_APERTURA_CAJA] + 
            df_arqueo['Efectivo Real'] -  # Ya incluye el Ingreso de Dinero sumado arriba
            df_arqueo['Pagos en Efectivo']
        )
        
        df_arqueo['Diferencia Efectivo'] = df_arqueo[COL_RECUENTO_EFECTIVO] - df_arqueo['Recuento sistema efectivo']
        df_arqueo['% diferencia efectivo'] = (df_arqueo['Diferencia Efectivo'] / df_arqueo['Efectivo Real']).replace([np.inf, -np.inf], 0).fillna(0)
        df_arqueo['Estado Diferencia Efectivo'] = df_arqueo['Diferencia Efectivo'].apply(clasificar_estado_diferencia)


        df_arqueo['Ventas Sistema MP Correcto'] = (
            df_arqueo['Ventas Sistema MP Real'] + 
            df_arqueo['Conciliado Efectivo, Sistema'] + 
            df_arqueo['Conciliado CtaCte, Sistema']
        )
        # =========================================================================

        df_arqueo['Intervalo ventas'] = df_arqueo['Hs Ultima Venta Sistema'] - df_arqueo['Hs Primera Venta Sistema']
        df_arqueo['Hs Ap Vs 1er venta'] = df_arqueo['Hs Primera Venta Sistema'] - df_arqueo['Apertura']
        df_arqueo['Hs Ultima vta Vs cierre'] = df_arqueo['Cierre'] - df_arqueo['Hs Ultima Venta Sistema']
        

        df_arqueo['Numerador_Promedios'] = (
            df_arqueo['Ventas MP Real'] + 
            df_arqueo['Efectivo Real'] + 
            df_arqueo['Cta Cte Real'] - 
            df_arqueo['Propina']
        )

        df_arqueo['Comensales promedio'] = (df_arqueo['Numerador_Promedios'] / df_arqueo[COL_COMENSALES].replace(0, np.nan)).fillna(0)
        df_arqueo['Ticket Promedio'] = (df_arqueo['Numerador_Promedios'] / df_arqueo['Cantidad de ticket'].replace(0, np.nan)).fillna(0)
        
        df_arqueo['% diferencia MP'] = (df_arqueo['Diferencia MP sistema vs real'] / df_arqueo['Ventas MP Real']).replace([np.inf, -np.inf], 0).fillna(0)
        
        df_arqueo['% diferencia facturacion'] = (df_arqueo['Diferencia de facturacion'] / df_arqueo['Ideal facturacion']).replace([np.inf, -np.inf], 0).fillna(0)
        
        df_arqueo['Estado Dif MP'] = df_arqueo['Diferencia MP sistema vs real'].apply(clasificar_estado_diferencia)
        # --- CORREGIDO: Se eliminaron Estado Dif Getnet y % diferencia Getnet ---

        cols_duracion_fix = ['Horas Trabajadas', 'Intervalo ventas', 'Hs Ap Vs 1er venta', 'Hs Ultima vta Vs cierre']
        for col in cols_duracion_fix:
            if col in df_arqueo.columns:
                df_arqueo[col] = df_arqueo[col].dt.total_seconds().div(86400).fillna(0)

        # ----------------------------------------------------------------------------------
        # COLUMNAS FINALES 
        # ----------------------------------------------------------------------------------
        cols_final = [
            'Fecha', 'Dia', 'Apertura', 'Hora Apertura', 'Cierre', 'Hora Cierre', 
            'Horas Trabajadas', 'TURNO', 'Encargado',
            'Hs Primera Venta Sistema', 'Hs Ultima Venta Sistema', 'Intervalo ventas', 'Hs Ap Vs 1er venta', 'Hs Ultima vta Vs cierre', 
            'Ventas Totales', 'Cantidad de comensales', 'Comensales promedio', 'Cantidad de ticket', 'Ticket Promedio',
            'Propina', 'Ventas MP Real', 'Ventas Sistema MP Real', 'Ventas Sistema MP Correcto',
            
            'Conciliado MP REAL', 'No conciliado MP REAL',
            'Conciliado Sistema MP', 'No conciiado Sistema MP', 'Diferencia MP sistema vs real', '% diferencia MP', 'Estado Dif MP', 
            'Efectivo Total', 'Conciliado Efectivo, Sistema', 'Efectivo Real', 
            'APERTURA CAJA Efectivo', 
            'Ingreso de Dinero', 
            'Pagos en Efectivo', 
            'Recuento Efectivo', 
            'Recuento sistema efectivo', 
            'Diferencia Efectivo',        
            '% diferencia efectivo',        
            'Estado Diferencia Efectivo', 
            'Cta Cte Total', 'Conciliado CtaCte, Sistema', 'Cta Cte Real', 
            'Otros',
            'Descuentos', 'Ventas facturadas', 'Ideal facturacion', 'Diferencia de facturacion', '% diferencia facturacion'
        ] + col_horas
        
        df_final = df_arqueo[[c for c in cols_final if c in df_arqueo.columns]]

        print(f"\n5. Abriendo Excel en segundo plano para actualizar: {os.path.basename(ruta_salida)}")
        
        app = xw.App(visible=False)
        try:
            wb = app.books.open(ruta_salida)
            
            try: ws = wb.sheets[NOMBRE_HOJA_DESTINO]
            except: ws = wb.sheets.add(NOMBRE_HOJA_DESTINO)
            
            print("   Limpiando hoja completa...")
            ws.clear() 

            ws.range('A1').options(index=False).value = df_final.columns.values.reshape(1, -1)
            
            if len(df_final) > 0:
                ws.range('A2').options(index=False).value = df_final.values
            
            print("   Aplicando formatos LOCALES...")
            header_range = ws.range(f'A1:{get_column_letter(len(df_final.columns))}1')
            header_range.api.Font.Bold = True
            
            total_rows = 1 + len(df_final)
            
            for col_idx, col_name in enumerate(df_final.columns, 1):
                col_letter = get_column_letter(col_idx)
                if total_rows > 1:
                    data_range = ws.range(f'{col_letter}2:{col_letter}{total_rows}')
                    
                    if col_name in COLS_FORMATO_PORCENTAJE:
                        data_range.api.NumberFormatLocal = "0,00%"
                    elif col_name in COLS_FORMATO_NUMERO:
                        data_range.api.NumberFormatLocal = "#.##0,00"
                    elif col_name == 'Fecha':
                        data_range.api.NumberFormatLocal = "dd/mm/aaaa"
                    elif col_name in ['Apertura', 'Cierre']:
                        data_range.api.NumberFormatLocal = "dd/mm/aaaa hh:mm:ss"
                    elif 'Hora' in col_name or 'Hs' in col_name or 'Intervalo' in col_name:
                        data_range.api.NumberFormatLocal = "hh:mm:ss"
                    elif col_name in col_horas:
                        data_range.api.NumberFormatLocal = "#.##0,00"
            
            ws.autofit()
            wb.save()
            print(f"\n¡ÉXITO! El archivo {os.path.basename(ruta_salida)} ha sido actualizado.")
            
        except Exception as e:
            print(f"Error con xlwings: {e}")
            traceback.print_exc()
        finally:
            try: wb.close()
            except: pass
            try: app.quit()
            except: pass

    except Exception as e:
        print(f"\n--- ERROR GENERAL --- {e}")
        traceback.print_exc()
    
    input("\nPresiona ENTER para finalizar...")

if __name__ == "__main__":
    generar_arqueo_por_turno()

   GENERADOR DE ARQUEO POR TURNO (V12 - SIN GETNET)

--> Esperando selección de: 1. Selecciona el archivo TURNOS PROCESADOS...
   V Seleccionado: Turnos_Procesados_playita.xlsx
--> Esperando selección de: 2. Selecciona el archivo RESULTADO CONCILIACION...
   V Seleccionado: Resultado_Conciliacion_playita.xlsx
--> Esperando selección de: 3. Selecciona el archivo INFORMES PARADOR (Ventas Sistema)...
   V Seleccionado: informes_playita.xlsx
--> Esperando selección de: 4. Selecciona el archivo con la hoja CAJA ADICION (Macro)...
   V Seleccionado: ARQUEO PRUEBA JOSE.xlsm

PASO 2: Selecciona el archivo de DESTINO.
--> Esperando selección de: 5. Selecciona el archivo ARQUEO FINAL (Excel .xlsm)...
   V Seleccionado: ARQUEO PRUEBA JOSE.xlsm

3. Leyendo archivos de datos...
   -> Datos cargados. Procesando...
4. Realizando cruces y cálculos...

5. Abriendo Excel en segundo plano para actualizar: ARQUEO PRUEBA JOSE.xlsm
   Limpiando hoja completa...
   Aplicando formatos LOCALES...

¡ÉXITO! El a